In [2]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
import math
import random
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)

## part a

In [5]:
G = nx.karate_club_graph()

nodes = list(G.nodes())

random.seed(42)

starting_nodes = random.sample(nodes, 3)

[7, 1, 17]


In [36]:
pairs = []

for i in starting_nodes:
    ending_nodes = [v for v in nodes if v != i] 
    j = random.choice(ending_nodes)
    pairs.append((i, j))

print(pairs)

[(7, 5), (1, 12), (17, 4)]


In [37]:
walkers = []

for idx, (i,j) in enumerate(pairs):
    walker = {
        'id': idx,
        'position': i, 
        'target': j,
        'steps': 0
    }

    walkers.append(walker)

print(walkers)

[{'id': 0, 'position': 7, 'target': 5, 'steps': 0}, {'id': 1, 'position': 1, 'target': 12, 'steps': 0}, {'id': 2, 'position': 17, 'target': 4, 'steps': 0}]


In [38]:
def random_neighbors(G, u):
    neighbor = random.choice(list(nx.neighbors(G, u)))
    return neighbor

In [55]:
def step_all_walkers(G, walkers):
    communite_time = []
    for w in walkers:
        while w['position'] != w['target']:
            w['position'] = random_neighbors(G, w['position'])
            w['steps'] += 1

        communite_time.append(w['steps'])
    return communite_time

In [56]:
communite_times = step_all_walkers(G, walkers)
print(communite_times)

[45, 34, 58]


## part b

In [42]:
from itertools import combinations

In [41]:
n = 500
p = 8 / 500
m = 4

G_ER = nx.erdos_renyi_graph(n, p, seed = 42)
G_BA = nx.barabasi_albert_graph(n, m, seed = 42)

In [63]:
ER_nodes = list(G_ER.nodes())
BA_nodes = list(G_BA.nodes())

all_pairs_ER = list(combinations(ER_nodes, 2))
all_pairs_BA = list(combinations(BA_nodes, 2))

ER_pairs = random.sample(all_pairs_ER, 300)
BA_pairs = random.sample(all_pairs_BA, 300)
print(ER_pairs)

[(134, 208), (85, 215), (344, 428), (190, 475), (129, 222), (87, 442), (182, 367), (205, 220), (16, 47), (200, 206), (263, 484), (26, 384), (466, 480), (276, 476), (57, 446), (207, 223), (58, 429), (75, 113), (223, 263), (21, 394), (43, 75), (67, 285), (47, 458), (175, 426), (20, 69), (42, 463), (0, 351), (122, 171), (137, 139), (237, 298), (193, 247), (144, 177), (83, 245), (8, 323), (64, 485), (82, 248), (246, 290), (80, 377), (244, 276), (344, 355), (138, 240), (19, 39), (236, 299), (65, 306), (414, 418), (75, 101), (292, 333), (294, 370), (206, 455), (191, 332), (223, 362), (302, 425), (430, 473), (54, 468), (127, 479), (31, 71), (172, 427), (62, 482), (217, 257), (40, 389), (392, 488), (75, 242), (318, 388), (38, 423), (19, 69), (15, 452), (45, 328), (295, 347), (88, 324), (193, 408), (268, 482), (316, 437), (182, 443), (409, 497), (82, 314), (132, 471), (33, 394), (143, 373), (236, 469), (87, 273), (242, 361), (120, 138), (450, 484), (77, 264), (155, 343), (170, 482), (153, 157),

In [88]:
def first_passage_time(G, start, end, max_steps = 10_000):
    curr = start
    step = 0
    
    if curr == end:
        return 0

    while curr != end and step < max_steps:
        curr = random_neighbors(G, curr)
        step += 1

    return step

def commute_time(G, i, j, n_trials):
    total_time = 0
    for _ in range(n_trials):
        t_ij = first_passage_time(G, i, j)
        t_ji = first_passage_time(G, j, i)
        total_time += (t_ij + t_ji)
    return total_time/n_trials

In [82]:
def compute_mean_commute(G, sampled_pairs, n_trials=100):
    total_commute = 0
    for (i, j) in sampled_pairs:
        total_commute += commute_time(G, i, j, n_trials=n_trials)
    return total_commute / len(sampled_pairs)

In [83]:
def sample_pairs(G, num_pairs=300):
    nodes = list(G.nodes())
    all_pairs = list(combinations(nodes, 2))
    return random.sample(all_pairs, num_pairs)

In [77]:
def average_ER_over_10_samples(num_pairs=300, n_trials=100):
    N = 500
    p = 8 / (N - 1)
    results = []

    for s in range(10):
        print(f"\n[ER graph sample {s+1}/10]")
        G_ER = nx.gnp_random_graph(N, p)

        # ensure connectivity
        if not nx.is_connected(G_ER):
            G_ER = G_ER.subgraph(max(nx.connected_components(G_ER), key=len)).copy()

        pairs = sample_pairs(G_ER, num_pairs=num_pairs)
        mean_commute = compute_mean_commute(G_ER, pairs, n_trials=n_trials)
        results.append(mean_commute)
        print(f"Mean commute time for this ER graph ≈ {mean_commute:.2f}")

    return sum(results) / len(results)


def average_BA_over_10_samples(num_pairs=300, n_trials=100):
    N = 500
    m = 4
    results = []

    for s in range(10):
        print(f"\n[BA graph sample {s+1}/10]")
        G_BA = nx.barabasi_albert_graph(N, m)

        pairs = sample_pairs(G_BA, num_pairs=num_pairs)
        mean_commute = compute_mean_commute(G_BA, pairs, n_trials=n_trials)
        results.append(mean_commute)
        print(f"Mean commute time for this BA graph ≈ {mean_commute:.2f}")

    return sum(results) / len(results)


In [89]:
avg_ER = average_ER_over_10_samples()
print("\nFinal average commute time for ER graphs:", avg_ER)

avg_BA = average_BA_over_10_samples()
print("\nFinal average commute time for BA graphs:", avg_BA)



[ER graph sample 1/10]
Mean commute time for this ER graph ≈ 1341.60

[ER graph sample 2/10]
Mean commute time for this ER graph ≈ 1357.89

[ER graph sample 3/10]
Mean commute time for this ER graph ≈ 1352.69

[ER graph sample 4/10]
Mean commute time for this ER graph ≈ 1359.60

[ER graph sample 5/10]
Mean commute time for this ER graph ≈ 1394.21

[ER graph sample 6/10]
Mean commute time for this ER graph ≈ 1336.34

[ER graph sample 7/10]
Mean commute time for this ER graph ≈ 1378.63

[ER graph sample 8/10]
Mean commute time for this ER graph ≈ 1346.48

[ER graph sample 9/10]
Mean commute time for this ER graph ≈ 1361.10

[ER graph sample 10/10]
Mean commute time for this ER graph ≈ 1328.31

Final average commute time for ER graphs: 1355.6832299999999

[BA graph sample 1/10]
Mean commute time for this BA graph ≈ 1595.73

[BA graph sample 2/10]
Mean commute time for this BA graph ≈ 1556.85

[BA graph sample 3/10]
Mean commute time for this BA graph ≈ 1617.20

[BA graph sample 4/10]
Mea